# Drive Revision Timestamp Pairing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/fix-worksheet-rename-detection-XrDXm/notebooks/drive-revision-pairs.ipynb)

Uses the Google Drive Revisions API (v3) to list **coarse-grained** revision
checkpoints for the UBL source spreadsheets. Each Drive revision has a
timestamp, which lets us find **natural (library, documents) pairs** — the
actual state that existed at each point in time.

## Why this matters

The internal Sheets revision count is huge (library: ~2005, documents: ~2204),
giving ~1M+ arbitrary integer combinations. But the Drive Revisions API returns
far fewer checkpoints (~25–200 per sheet), each timestamped. For each documents
revision at time `T_doc`, we find the library revision with the largest
timestamp `T_lib ≤ T_doc` — giving us **exactly one natural pair** per
documents revision.

## Output

Saves to Google Drive `ubl-gc-revisions/drive-revision-pairs/`:
- `drive-revisions-metadata.json` — raw revision lists for all sheets
- `temporal-pairs.json` — computed (lib, doc) pairs with timestamps
- ODS + GC files for each unique pair (optional Phase 2)

## Sheets covered

| Sheet | ID | Notes |
|-------|----|-------|
| UBL 2.5 Library | `18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY` | Main |
| UBL 2.5 Documents | `1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg` | Main |
| UBL 2.5 Signature | `1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g` | Stable, included for completeness |
| UBL 2.4 Library | `1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs` | Previous version |
| UBL 2.4 Documents | `1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y` | Previous version |

In [ ]:
# === Step 0: Auth ===
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive.readonly']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
TOKEN_EXPIRES = creds.expiry

def refresh_token():
    """Refresh token if expired or close to expiry."""
    global TOKEN, TOKEN_EXPIRES
    import datetime
    if TOKEN_EXPIRES and TOKEN_EXPIRES < datetime.datetime.utcnow() + datetime.timedelta(minutes=5):
        creds.refresh(AuthRequest())
        TOKEN = creds.token
        TOKEN_EXPIRES = creds.expiry
        print(f'Token refreshed, expires {TOKEN_EXPIRES}')
    return TOKEN

print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')
print(f'Expires: {TOKEN_EXPIRES}')

In [ ]:
# === Step 1: Mount Drive & create output directory ===
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_BASE = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR = DRIVE_BASE / 'drive-revision-pairs'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output: {DRIVE_DIR}')

In [ ]:
# === Step 2: Configuration ===
import json, time, bisect
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from datetime import datetime, timezone
from collections import defaultdict

# All known UBL source spreadsheets
SHEETS = {
    # UBL 2.5
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
    'ubl25_signature': '1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g',
    # UBL 2.4
    'ubl24_library':   '1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs',
    'ubl24_documents': '1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y',
}

# Pairing groups: which sheets form (library, documents) pairs
PAIR_GROUPS = {
    'ubl25': {
        'library': 'ubl25_library',
        'documents': 'ubl25_documents',
        'signature': 'ubl25_signature',  # included for timestamp reference
    },
    'ubl24': {
        'library': 'ubl24_library',
        'documents': 'ubl24_documents',
    },
}

API_DELAY = 0.5  # seconds between API calls


def api_get_json(url):
    """Authenticated JSON GET with retry + exponential backoff."""
    token = refresh_token()
    headers = {'Authorization': f'Bearer {token}'}
    for attempt in range(4):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=30) as resp:
                return resp.status, json.loads(resp.read())
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  retry({e.code}, {wait}s)...', end='')
                time.sleep(wait)
                continue
            return e.code, e.read().decode(errors='replace')
    return 0, 'max retries exceeded'


def parse_ts(ts_str):
    """Parse ISO 8601 timestamp to datetime."""
    if not ts_str:
        return None
    # Handle both 'Z' suffix and '+00:00'
    ts_str = ts_str.replace('Z', '+00:00')
    return datetime.fromisoformat(ts_str)


print('Configuration ready')
print(f'Sheets: {len(SHEETS)}')
for k, v in SHEETS.items():
    print(f'  {k}: {v[:12]}...')

## Phase 1: Fetch Drive Revision Metadata

Uses `drive/v3/files/{id}/revisions` to get all revision checkpoints.
Each revision has: `id`, `modifiedTime`, `lastModifyingUser`, `size`.

These are **coarser** than internal Sheets revisions — Google batches
many edits into a single Drive revision checkpoint.

In [ ]:
# === Step 3: Fetch all Drive revisions for every sheet ===

def get_all_drive_revisions(file_id, sheet_key):
    """Paginate through all Drive v3 revisions of a file."""
    all_revisions = []
    page_token = None

    while True:
        url = (
            f'https://www.googleapis.com/drive/v3/files/{file_id}/revisions'
            f'?pageSize=1000'
            f'&fields=nextPageToken,revisions(id,modifiedTime,'
            f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,size)'
        )
        if page_token:
            url += f'&pageToken={page_token}'

        status, data = api_get_json(url)
        if status != 200:
            print(f'  ERROR {status}: {str(data)[:200]}')
            break

        revisions = data.get('revisions', [])
        all_revisions.extend(revisions)

        page_token = data.get('nextPageToken')
        if not page_token:
            break
        time.sleep(API_DELAY)

    return all_revisions


# Fetch revisions for all sheets
all_metadata = {}

for sheet_key, file_id in SHEETS.items():
    print(f'Fetching {sheet_key} ({file_id[:12]}...)...', end=' ')
    revisions = get_all_drive_revisions(file_id, sheet_key)

    if revisions:
        first_ts = revisions[0].get('modifiedTime', '?')
        last_ts = revisions[-1].get('modifiedTime', '?')
        print(f'{len(revisions)} revisions ({first_ts[:10]} → {last_ts[:10]})')
    else:
        print('0 revisions (or error)')

    all_metadata[sheet_key] = {
        'file_id': file_id,
        'revision_count': len(revisions),
        'revisions': revisions,
    }
    time.sleep(API_DELAY)

print(f'\nTotal: {sum(m["revision_count"] for m in all_metadata.values())} Drive revisions across {len(SHEETS)} sheets')

In [ ]:
# === Step 4: Display revision timeline for each sheet ===

for sheet_key, meta in all_metadata.items():
    revisions = meta['revisions']
    print(f'\n{"=" * 70}')
    print(f'{sheet_key}: {len(revisions)} Drive revisions')
    print(f'{"=" * 70}')

    if not revisions:
        print('  (none)')
        continue

    # Group by date for readability
    by_date = defaultdict(list)
    for rev in revisions:
        ts = rev.get('modifiedTime', '')
        date = ts[:10] if ts else '?'
        by_date[date].append(rev)

    print(f'  Spans {len(by_date)} distinct dates')
    print(f'  First: {revisions[0].get("modifiedTime", "?")}')
    print(f'  Last:  {revisions[-1].get("modifiedTime", "?")}')
    print()

    # Show first few and last few with full detail
    show_revs = revisions[:5] + (['...'] if len(revisions) > 10 else []) + revisions[-5:] if len(revisions) > 10 else revisions
    for i, rev in enumerate(show_revs):
        if rev == '...':
            print(f'  ... ({len(revisions) - 10} more) ...')
            continue
        rid = rev.get('id', '?')
        ts = rev.get('modifiedTime', '?')
        user = rev.get('lastModifyingUser', {}).get('displayName', '?')
        size = rev.get('size', '?')
        print(f'  [{rid:>6s}] {ts}  {user}  ({size} bytes)')

## Phase 2: Compute Temporal Pairs

For each documents revision at timestamp `T_doc`, find the library revision
with the largest timestamp `T_lib ≤ T_doc` (the library state that was current
when that documents revision was saved).

This also works in reverse — for each library revision, find the documents
state at that time. We compute **both directions** and merge to get the full
set of natural temporal snapshots.

In [ ]:
# === Step 5: Compute temporal pairs ===

def compute_temporal_pairs(lib_revisions, doc_revisions, sig_revisions=None):
    """
    Given two lists of Drive revisions (each with 'id' and 'modifiedTime'),
    compute natural temporal pairs.

    For every revision in EITHER list, find the corresponding revision in the
    OTHER list that was current at that timestamp (latest revision with
    modifiedTime <= the target timestamp).

    Returns a list of unique (lib_rev_id, doc_rev_id, sig_rev_id, timestamp) tuples,
    sorted chronologically.
    """
    # Parse timestamps and build sorted lists
    lib_entries = []
    for rev in lib_revisions:
        ts = parse_ts(rev.get('modifiedTime'))
        if ts:
            lib_entries.append((ts, rev['id'], rev))
    lib_entries.sort(key=lambda x: x[0])
    lib_timestamps = [e[0] for e in lib_entries]

    doc_entries = []
    for rev in doc_revisions:
        ts = parse_ts(rev.get('modifiedTime'))
        if ts:
            doc_entries.append((ts, rev['id'], rev))
    doc_entries.sort(key=lambda x: x[0])
    doc_timestamps = [e[0] for e in doc_entries]

    sig_entries = []
    if sig_revisions:
        for rev in sig_revisions:
            ts = parse_ts(rev.get('modifiedTime'))
            if ts:
                sig_entries.append((ts, rev['id'], rev))
        sig_entries.sort(key=lambda x: x[0])
    sig_timestamps = [e[0] for e in sig_entries] if sig_entries else []

    def find_current(entries, timestamps, at_time):
        """Find the latest revision with timestamp <= at_time."""
        idx = bisect.bisect_right(timestamps, at_time) - 1
        if idx < 0:
            return None
        return entries[idx]

    # Collect all unique pairs
    seen_pairs = set()
    pairs = []

    def add_pair(lib_entry, doc_entry, sig_entry, trigger_ts, trigger_source):
        if lib_entry is None or doc_entry is None:
            return
        pair_key = (lib_entry[1], doc_entry[1])
        if pair_key in seen_pairs:
            return
        seen_pairs.add(pair_key)
        pairs.append({
            'lib_rev_id': lib_entry[1],
            'lib_timestamp': lib_entry[0].isoformat(),
            'lib_user': lib_entry[2].get('lastModifyingUser', {}).get('displayName', '?'),
            'doc_rev_id': doc_entry[1],
            'doc_timestamp': doc_entry[0].isoformat(),
            'doc_user': doc_entry[2].get('lastModifyingUser', {}).get('displayName', '?'),
            'sig_rev_id': sig_entry[1] if sig_entry else None,
            'sig_timestamp': sig_entry[0].isoformat() if sig_entry else None,
            'trigger_timestamp': trigger_ts.isoformat(),
            'trigger_source': trigger_source,
        })

    # Forward: for each doc revision, find current lib
    for doc_ts, doc_id, doc_rev in doc_entries:
        lib_match = find_current(lib_entries, lib_timestamps, doc_ts)
        sig_match = find_current(sig_entries, sig_timestamps, doc_ts) if sig_entries else None
        add_pair(lib_match, (doc_ts, doc_id, doc_rev), sig_match, doc_ts, 'doc_changed')

    # Reverse: for each lib revision, find current doc
    for lib_ts, lib_id, lib_rev in lib_entries:
        doc_match = find_current(doc_entries, doc_timestamps, lib_ts)
        sig_match = find_current(sig_entries, sig_timestamps, lib_ts) if sig_entries else None
        add_pair((lib_ts, lib_id, lib_rev), doc_match, sig_match, lib_ts, 'lib_changed')

    # Also: for each sig revision, find current lib + doc
    for sig_ts, sig_id, sig_rev in sig_entries:
        lib_match = find_current(lib_entries, lib_timestamps, sig_ts)
        doc_match = find_current(doc_entries, doc_timestamps, sig_ts)
        add_pair(lib_match, doc_match, (sig_ts, sig_id, sig_rev), sig_ts, 'sig_changed')

    # Sort by trigger timestamp
    pairs.sort(key=lambda p: p['trigger_timestamp'])
    return pairs


# Compute pairs for each version group
all_pairs = {}

for group_name, group_sheets in PAIR_GROUPS.items():
    lib_key = group_sheets['library']
    doc_key = group_sheets['documents']
    sig_key = group_sheets.get('signature')

    lib_revs = all_metadata.get(lib_key, {}).get('revisions', [])
    doc_revs = all_metadata.get(doc_key, {}).get('revisions', [])
    sig_revs = all_metadata.get(sig_key, {}).get('revisions', []) if sig_key else None

    print(f'\n{"=" * 70}')
    print(f'{group_name}: {len(lib_revs)} lib revs × {len(doc_revs)} doc revs'
          f'{f" × {len(sig_revs)} sig revs" if sig_revs else ""}')

    pairs = compute_temporal_pairs(lib_revs, doc_revs, sig_revs)
    all_pairs[group_name] = pairs

    print(f'→ {len(pairs)} unique temporal pairs')
    print(f'{"=" * 70}')

    brute_force = len(lib_revs) * len(doc_revs)
    if brute_force > 0:
        reduction = brute_force / max(len(pairs), 1)
        print(f'  Brute force would be: {brute_force:,} pairs')
        print(f'  Reduction factor: {reduction:,.0f}×')
    print()

    for i, pair in enumerate(pairs):
        ts = pair['trigger_timestamp'][:19]
        src = pair['trigger_source']
        lib_id = pair['lib_rev_id']
        doc_id = pair['doc_rev_id']
        sig_id = pair.get('sig_rev_id') or '-'
        lib_ts = pair['lib_timestamp'][:19]
        doc_ts = pair['doc_timestamp'][:19]
        print(f'  {i+1:3d}. [{ts}] lib={lib_id:>6s} ({lib_ts}) + doc={doc_id:>6s} ({doc_ts})'
              f' sig={sig_id:>6s}  ← {src}')

In [ ]:
# === Step 6: Cross-reference with known CI workflow runs (UBL 2.5) ===

# Known V1-V10 workflow run timestamps and their validated revision pairs
KNOWN_CI_RUNS = {
    'V1':  {'ts': '2025-11-17T10:42', 'lib': '1843', 'doc': '1793'},
    'V2':  {'ts': '2025-11-19T09:15', 'lib': '1843', 'doc': '1793'},
    'V3':  {'ts': '2025-11-20T13:50', 'lib': '1868', 'doc': '1803'},
    'V4':  {'ts': '2025-11-20T14:05', 'lib': '1868', 'doc': '1983'},
    'V5':  {'ts': '2025-12-03T09:00', 'lib': '1999', 'doc': '2190'},
    'V6':  {'ts': '2026-01-21T16:38', 'lib': '1999', 'doc': '2190'},
    'V7':  {'ts': '2026-01-21T17:01', 'lib': '2005', 'doc': '2190'},
    'V8':  {'ts': '2026-01-21T19:26', 'lib': '2005', 'doc': '2200'},
    'V9':  {'ts': '2026-02-09T14:42', 'lib': '2005', 'doc': '2204'},
    'V10': {'ts': '2026-02-09T14:46', 'lib': '2005', 'doc': '2204'},
}

if 'ubl25' in all_pairs:
    print('Cross-reference: Drive temporal pairs vs known CI run revision pairs')
    print('=' * 70)
    print()
    print('Known CI runs use INTERNAL Sheets revision IDs (e.g., 1843, 2005).')
    print('Drive API returns DRIVE revision IDs (may differ!).')
    print('This table helps map between the two systems.')
    print()

    ubl25_pairs = all_pairs['ubl25']

    for ci_name, ci_data in KNOWN_CI_RUNS.items():
        ci_ts = parse_ts(ci_data['ts'] + ':00+00:00')
        # Find the temporal pair closest to (but before) this CI run
        best_pair = None
        for pair in ubl25_pairs:
            pair_ts = parse_ts(pair['trigger_timestamp'])
            if pair_ts and pair_ts <= ci_ts:
                best_pair = pair
        if best_pair:
            print(f'  {ci_name} ({ci_data["ts"]}): '
                  f'sheets lib={ci_data["lib"]}/doc={ci_data["doc"]}  '
                  f'→ drive lib={best_pair["lib_rev_id"]}/doc={best_pair["doc_rev_id"]}')
        else:
            print(f'  {ci_name} ({ci_data["ts"]}): '
                  f'sheets lib={ci_data["lib"]}/doc={ci_data["doc"]}  '
                  f'→ no Drive pair found before this timestamp')
else:
    print('No UBL 2.5 pairs computed (sheet metadata may have failed)')

In [ ]:
# === Step 7: Save metadata to Drive ===

output = {
    '_provenance': {
        'description': 'Drive Revisions API v3 metadata + temporal pairing for UBL source sheets',
        'method': 'drive/v3/files/{id}/revisions → timestamp-based binary search pairing',
        'notebook': 'notebooks/drive-revision-pairs.ipynb',
        'fetched_at': datetime.now(timezone.utc).isoformat(),
    },
    'sheets': {},
    'temporal_pairs': {},
    'summary': {},
}

# Sheet metadata (with full revision lists)
for sheet_key, meta in all_metadata.items():
    output['sheets'][sheet_key] = {
        'file_id': meta['file_id'],
        'revision_count': meta['revision_count'],
        'first_revision': meta['revisions'][0] if meta['revisions'] else None,
        'last_revision': meta['revisions'][-1] if meta['revisions'] else None,
        'revisions': meta['revisions'],
    }

# Temporal pairs
for group_name, pairs in all_pairs.items():
    group_sheets = PAIR_GROUPS[group_name]
    lib_count = all_metadata.get(group_sheets['library'], {}).get('revision_count', 0)
    doc_count = all_metadata.get(group_sheets['documents'], {}).get('revision_count', 0)

    output['temporal_pairs'][group_name] = {
        'library_sheet': group_sheets['library'],
        'documents_sheet': group_sheets['documents'],
        'signature_sheet': group_sheets.get('signature'),
        'lib_drive_revision_count': lib_count,
        'doc_drive_revision_count': doc_count,
        'brute_force_pairs': lib_count * doc_count,
        'temporal_pairs_count': len(pairs),
        'reduction_factor': (lib_count * doc_count) / max(len(pairs), 1),
        'pairs': pairs,
    }

# Summary
output['summary'] = {
    'total_sheets': len(SHEETS),
    'total_drive_revisions': sum(m['revision_count'] for m in all_metadata.values()),
    'groups': {
        name: {
            'temporal_pairs': len(pairs),
            'brute_force': output['temporal_pairs'][name]['brute_force_pairs'],
            'reduction': f"{output['temporal_pairs'][name]['reduction_factor']:.0f}x",
        }
        for name, pairs in all_pairs.items()
    },
}

# Write full metadata
meta_path = DRIVE_DIR / 'drive-revisions-metadata.json'
meta_path.write_text(json.dumps(output, indent=2))
print(f'Full metadata: {meta_path} ({meta_path.stat().st_size:,} bytes)')

# Write just the temporal pairs (smaller, for quick consumption)
pairs_path = DRIVE_DIR / 'temporal-pairs.json'
pairs_only = {
    '_provenance': output['_provenance'],
    'summary': output['summary'],
    'temporal_pairs': {
        name: {
            'temporal_pairs_count': data['temporal_pairs_count'],
            'pairs': data['pairs'],
        }
        for name, data in output['temporal_pairs'].items()
    },
}
pairs_path.write_text(json.dumps(pairs_only, indent=2))
print(f'Pairs only:    {pairs_path} ({pairs_path.stat().st_size:,} bytes)')

print(f'\nSaved to: {DRIVE_DIR}')

## Phase 2 (Optional): Download ODS for Each Unique Pair

Uses Drive API v2 `exportLinks` to download revision-specific ODS files.
Only runs if you execute this cell — Phase 1 results are already saved above.

**Note:** This may take a while depending on how many pairs were found.
Each download requires 2 API calls (get exportLinks, download ODS) per sheet.

In [ ]:
# === Step 8 (Optional): Download ODS for each temporal pair ===

import hashlib, zipfile, io

ODS_MIME = 'application/x-vnd.oasis.opendocument.spreadsheet'
ODS_MIME_ALT = 'application/vnd.oasis.opendocument.spreadsheet'


def api_download(url):
    """Authenticated binary GET with retry."""
    token = refresh_token()
    headers = {'Authorization': f'Bearer {token}'}
    for attempt in range(4):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                return resp.status, resp.read()
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  retry({e.code}, {wait}s)...', end='')
                time.sleep(wait)
                continue
            return e.code, None
    return 0, None


def get_v2_export_links(file_id, revision_id):
    """Get exportLinks from Drive API v2 for a specific revision."""
    url = f'https://www.googleapis.com/drive/v2/files/{file_id}/revisions/{revision_id}'
    status, data = api_get_json(url)
    if status == 200 and isinstance(data, dict):
        return data.get('exportLinks', {})
    print(f'    ERROR getting v2 exportLinks: HTTP {status}')
    return {}


def download_ods_via_v2(file_id, revision_id):
    """Download ODS at a specific revision using Drive API v2 exportLinks."""
    links = get_v2_export_links(file_id, revision_id)
    if not links:
        return None
    ods_url = links.get(ODS_MIME) or links.get(ODS_MIME_ALT)
    if not ods_url:
        print(f'    No ODS format in exportLinks. Available: {list(links.keys())}')
        return None
    status, data = api_download(ods_url)
    if status == 200 and data and len(data) > 500:
        return data
    print(f'    ODS download failed: HTTP {status}')
    return None


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS (ZIP) and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return hashlib.sha256(ods_bytes).hexdigest()  # fallback: hash raw bytes


# === Which group to download? ===
DOWNLOAD_GROUP = 'ubl25'  # Change to 'ubl24' for UBL 2.4

group = PAIR_GROUPS[DOWNLOAD_GROUP]
pairs = all_pairs.get(DOWNLOAD_GROUP, [])
lib_file_id = SHEETS[group['library']]
doc_file_id = SHEETS[group['documents']]
sig_file_id = SHEETS.get(group.get('signature', ''), '')

print(f'Downloading ODS for {len(pairs)} temporal pairs ({DOWNLOAD_GROUP})')
print(f'  Library:   {lib_file_id[:12]}...')
print(f'  Documents: {doc_file_id[:12]}...')
if sig_file_id:
    print(f'  Signature: {sig_file_id[:12]}...')
print()

ods_dir = DRIVE_DIR / f'{DOWNLOAD_GROUP}-ods'
ods_dir.mkdir(exist_ok=True)

# Track unique content hashes to avoid redundant downloads
seen_lib_revs = {}   # rev_id → ods_path
seen_doc_revs = {}   # rev_id → ods_path
seen_sig_revs = {}   # rev_id → ods_path
download_log = []

for i, pair in enumerate(pairs):
    lib_rev = pair['lib_rev_id']
    doc_rev = pair['doc_rev_id']
    sig_rev = pair.get('sig_rev_id')
    trigger = pair['trigger_timestamp'][:19]

    print(f'[{i+1}/{len(pairs)}] {trigger}: lib={lib_rev} doc={doc_rev}', end='')

    entry = {
        'pair_index': i,
        'lib_rev_id': lib_rev,
        'doc_rev_id': doc_rev,
        'sig_rev_id': sig_rev,
        'trigger_timestamp': pair['trigger_timestamp'],
    }

    # Download library ODS (skip if same rev already downloaded)
    if lib_rev in seen_lib_revs:
        entry['lib_ods'] = seen_lib_revs[lib_rev]
        entry['lib_status'] = 'cached'
    else:
        lib_path = ods_dir / f'lib-rev-{lib_rev}.ods'
        if lib_path.exists() and lib_path.stat().st_size > 500:
            entry['lib_ods'] = str(lib_path)
            entry['lib_hash'] = ods_content_hash(lib_path.read_bytes())
            entry['lib_size'] = lib_path.stat().st_size
            entry['lib_status'] = 'exists'
        else:
            print(f' ↓lib', end='')
            ods_data = download_ods_via_v2(lib_file_id, lib_rev)
            if ods_data:
                lib_path.write_bytes(ods_data)
                entry['lib_ods'] = str(lib_path)
                entry['lib_hash'] = ods_content_hash(ods_data)
                entry['lib_size'] = len(ods_data)
                entry['lib_status'] = 'downloaded'
            else:
                entry['lib_status'] = 'failed'
            time.sleep(API_DELAY)
        seen_lib_revs[lib_rev] = entry.get('lib_ods')

    # Download documents ODS
    if doc_rev in seen_doc_revs:
        entry['doc_ods'] = seen_doc_revs[doc_rev]
        entry['doc_status'] = 'cached'
    else:
        doc_path = ods_dir / f'doc-rev-{doc_rev}.ods'
        if doc_path.exists() and doc_path.stat().st_size > 500:
            entry['doc_ods'] = str(doc_path)
            entry['doc_hash'] = ods_content_hash(doc_path.read_bytes())
            entry['doc_size'] = doc_path.stat().st_size
            entry['doc_status'] = 'exists'
        else:
            print(f' ↓doc', end='')
            ods_data = download_ods_via_v2(doc_file_id, doc_rev)
            if ods_data:
                doc_path.write_bytes(ods_data)
                entry['doc_ods'] = str(doc_path)
                entry['doc_hash'] = ods_content_hash(ods_data)
                entry['doc_size'] = len(ods_data)
                entry['doc_status'] = 'downloaded'
            else:
                entry['doc_status'] = 'failed'
            time.sleep(API_DELAY)
        seen_doc_revs[doc_rev] = entry.get('doc_ods')

    # Download signature ODS (if applicable)
    if sig_rev and sig_file_id:
        if sig_rev in seen_sig_revs:
            entry['sig_ods'] = seen_sig_revs[sig_rev]
            entry['sig_status'] = 'cached'
        else:
            sig_path = ods_dir / f'sig-rev-{sig_rev}.ods'
            if sig_path.exists() and sig_path.stat().st_size > 500:
                entry['sig_ods'] = str(sig_path)
                entry['sig_hash'] = ods_content_hash(sig_path.read_bytes())
                entry['sig_size'] = sig_path.stat().st_size
                entry['sig_status'] = 'exists'
            else:
                print(f' ↓sig', end='')
                ods_data = download_ods_via_v2(sig_file_id, sig_rev)
                if ods_data:
                    sig_path.write_bytes(ods_data)
                    entry['sig_ods'] = str(sig_path)
                    entry['sig_hash'] = ods_content_hash(ods_data)
                    entry['sig_size'] = len(ods_data)
                    entry['sig_status'] = 'downloaded'
                else:
                    entry['sig_status'] = 'failed'
                time.sleep(API_DELAY)
            seen_sig_revs[sig_rev] = entry.get('sig_ods')

    download_log.append(entry)
    print()  # newline

# Save download log
log_path = DRIVE_DIR / f'{DOWNLOAD_GROUP}-download-log.json'
log_path.write_text(json.dumps(download_log, indent=2))

n_lib = len(seen_lib_revs)
n_doc = len(seen_doc_revs)
n_sig = len(seen_sig_revs)
print(f'\nDone: {n_lib} unique lib + {n_doc} unique doc + {n_sig} unique sig ODS files')
print(f'Download log: {log_path}')

## Phase 3 (Optional): Convert ODS → GenericCode

Requires Java (available by default in Colab) and the Saxon + Crane tools.
Downloads tool files from GitHub, then converts each unique ODS pair to `.gc`.

**Only run this if you also ran Phase 2 (ODS downloads).**

In [ ]:
# === Step 9 (Optional): Fetch tools + convert ODS → GC ===

import subprocess, shutil, tempfile, gzip

TOOLS_DIR = Path('/content/tools')
TOOLS_DIR.mkdir(exist_ok=True)
(TOOLS_DIR / 'support').mkdir(exist_ok=True)

BRANCH = 'claude/fix-worksheet-rename-detection-XrDXm'
RAW = f'https://raw.githubusercontent.com/kduvekot/ubl-gc/{BRANCH}'
CRANE = 'history/tools/Crane-ods2obdgc'

TOOL_URLS = {
    'saxon9he.jar':               f'{RAW}/history/tools/saxon9he/saxon9he.jar',
    'Crane-ods2obdgc.xsl':        f'{RAW}/{CRANE}/Crane-ods2obdgc.xsl',
    'support/gcExportSubset.xsl':  f'{RAW}/{CRANE}/support/gcExportSubset.xsl',
    'support/odsCommon.xsl':       f'{RAW}/{CRANE}/support/odsCommon.xsl',
    'massageModelName.xml':        f'{RAW}/work-sheets/scripts/massageModelName.xml',
    'gc2endorsed.xsl':             f'{RAW}/work-sheets/scripts/gc2endorsed.xsl',
}

for name, url in TOOL_URLS.items():
    dest = TOOLS_DIR / name
    if dest.exists() and dest.stat().st_size > 100:
        continue
    print(f'  Downloading {name}...', end=' ')
    r = subprocess.run(['wget', '-q', '-O', str(dest), url],
                       capture_output=True, timeout=60)
    print(f'{dest.stat().st_size:,} bytes' if dest.exists() else 'FAILED')

SAXON_JAR   = str(TOOLS_DIR / 'saxon9he.jar')
CRANE_XSL   = str(TOOLS_DIR / 'Crane-ods2obdgc.xsl')
MASSAGE_XML = str(TOOLS_DIR / 'massageModelName.xml')
GC2ENDORSED = str(TOOLS_DIR / 'gc2endorsed.xsl')

SHEET_REGEX = r'^([Ll]($|[^o].*|o($|[^g].*|g($|[^s].*))))|^[^Ll].*'

# Verify Java
!java -version 2>&1 | head -1
print('Tools ready!')

In [ ]:
# === Step 10 (Optional): Convert each unique pair to GC ===

import re

PH_STAGE = '@@STAGE@@'
PH_stage = '@@stage@@'


def make_ident_xml(tmpdir, endorsed=False):
    sfx  = '-Endorsed' if endorsed else ''
    nsfx = ' Endorsed' if endorsed else ''
    usfx = ':ENDORSED' if endorsed else ''
    fsfx = '-Endorsed' if endorsed else ''
    xml = (
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        '<Identification>\n'
        f'  <ShortName>UBL-2.5-{PH_STAGE}{sfx}</ShortName>\n'
        f'  <LongName>UBL 2.5 {PH_STAGE}{nsfx} Business Entity Summary</LongName>\n'
        '  <Version>2.5</Version>\n'
        f'  <CanonicalUri>urn:oasis:names:specification:ubl:BIE{usfx}</CanonicalUri>\n'
        f'  <CanonicalVersionUri>urn:oasis:names:specification:ubl:BIE{usfx}:2.5</CanonicalVersionUri>\n'
        f'  <LocationUri>http://docs.oasis-open.org/ubl/{PH_stage}-UBL-2.5/mod/UBL-Entities-2.5{fsfx}.gc</LocationUri>\n'
        '  <Agency>\n'
        '     <LongName xml:lang="en">OASIS Universal Business Language</LongName>\n'
        '     <Identifier>UBL</Identifier>\n'
        '  </Agency>\n'
        '</Identification>'
    )
    fname = 'ident-UBL-Endorsed.xml' if endorsed else 'ident-UBL.xml'
    path = os.path.join(tmpdir, fname)
    with open(path, 'w') as f:
        f.write(xml)
    return path


def run_saxon(args):
    cmd = ['java', '-jar', SAXON_JAR] + args
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    return result.returncode == 0, result.stderr


def gc_data_hash(gc_bytes):
    text = gc_bytes.decode('utf-8')
    stripped = re.sub(
        r'<Identification>.*?</Identification>\s*',
        '', text, count=1, flags=re.DOTALL
    )
    return hashlib.sha256(stripped.encode('utf-8')).hexdigest()


def convert_to_gc(lib_ods_path, doc_ods_path):
    tmpdir = tempfile.mkdtemp()
    try:
        shutil.copy2(str(lib_ods_path), os.path.join(tmpdir, 'UBL-Library-Google.ods'))
        shutil.copy2(str(doc_ods_path), os.path.join(tmpdir, 'UBL-Documents-Google.ods'))
        shutil.copy2(MASSAGE_XML, os.path.join(tmpdir, 'massageModelName.xml'))

        ods_list = f'{tmpdir}/UBL-Library-Google.ods,{tmpdir}/UBL-Documents-Google.ods'
        make_ident_xml(tmpdir, endorsed=False)
        make_ident_xml(tmpdir, endorsed=True)

        entities_out = os.path.join(tmpdir, 'entities.gc')
        endorsed_out = os.path.join(tmpdir, 'endorsed.gc')
        raw_endorsed = os.path.join(tmpdir, 'raw-endorsed.gc')

        ok, stderr = run_saxon([
            f'-xsl:{CRANE_XSL}', f'-o:{entities_out}', '-it:ods-uri',
            f'ods-uri={ods_list}',
            f'identification-uri={tmpdir}/ident-UBL.xml',
            f'included-sheet-name-regex={SHEET_REGEX}',
            f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
        ])
        if not ok:
            return None, None, f'entities: {stderr[:500]}'

        ok, stderr = run_saxon([
            f'-xsl:{CRANE_XSL}', f'-o:{raw_endorsed}', '-it:ods-uri',
            f'ods-uri={ods_list}',
            f'identification-uri={tmpdir}/ident-UBL-Endorsed.xml',
            f'included-sheet-name-regex={SHEET_REGEX}',
            f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
        ])
        if not ok:
            return None, None, f'endorsed-raw: {stderr[:500]}'

        ok, stderr = run_saxon([
            f'-o:{endorsed_out}', f'-s:{raw_endorsed}',
            f'-xsl:{GC2ENDORSED}',
        ])
        if not ok:
            return None, None, f'endorsed-filter: {stderr[:500]}'

        return (Path(entities_out).read_bytes(),
                Path(endorsed_out).read_bytes(),
                None)
    except Exception as e:
        return None, None, f'exception: {e}'
    finally:
        shutil.rmtree(tmpdir)


# === Convert all unique pairs ===
gc_dir = DRIVE_DIR / f'{DOWNLOAD_GROUP}-gc'
gc_dir.mkdir(exist_ok=True)

# Identify unique (lib_rev, doc_rev) pairs from download log
unique_pairs = []
seen = set()
for entry in download_log:
    key = (entry['lib_rev_id'], entry['doc_rev_id'])
    if key in seen:
        continue
    seen.add(key)
    if entry.get('lib_ods') and entry.get('doc_ods'):
        unique_pairs.append(entry)

print(f'Converting {len(unique_pairs)} unique (lib, doc) pairs to GenericCode')
print()

gc_results = []

for i, entry in enumerate(unique_pairs):
    lib_rev = entry['lib_rev_id']
    doc_rev = entry['doc_rev_id']
    lib_ods = Path(entry['lib_ods'])
    doc_ods = Path(entry['doc_ods'])

    prefix = f'lib{lib_rev}-doc{doc_rev}'
    ent_path = gc_dir / f'{prefix}.entities.gc.gz'
    end_path = gc_dir / f'{prefix}.endorsed.gc.gz'

    if ent_path.exists() and end_path.exists():
        print(f'  [{i+1}/{len(unique_pairs)}] {prefix} — already converted')
        gc_results.append({
            'lib_rev': lib_rev, 'doc_rev': doc_rev,
            'status': 'cached',
            'entities_path': str(ent_path),
            'endorsed_path': str(end_path),
        })
        continue

    print(f'  [{i+1}/{len(unique_pairs)}] {prefix}...', end=' ')
    ent_bytes, end_bytes, err = convert_to_gc(lib_ods, doc_ods)

    if ent_bytes and end_bytes:
        ent_gz = gzip.compress(ent_bytes, compresslevel=6)
        end_gz = gzip.compress(end_bytes, compresslevel=6)
        ent_path.write_bytes(ent_gz)
        end_path.write_bytes(end_gz)

        ent_hash = gc_data_hash(ent_bytes)
        end_hash = gc_data_hash(end_bytes)
        print(f'OK  entities={ent_hash[:12]}  endorsed={end_hash[:12]}')

        gc_results.append({
            'lib_rev': lib_rev, 'doc_rev': doc_rev,
            'status': 'ok',
            'entities_path': str(ent_path),
            'endorsed_path': str(end_path),
            'entities_data_hash': ent_hash,
            'endorsed_data_hash': end_hash,
            'entities_size': len(ent_bytes),
            'endorsed_size': len(end_bytes),
        })
    else:
        print(f'FAILED: {err}')
        gc_results.append({
            'lib_rev': lib_rev, 'doc_rev': doc_rev,
            'status': 'failed', 'error': err,
        })

# Save GC results
gc_results_path = DRIVE_DIR / f'{DOWNLOAD_GROUP}-gc-results.json'
gc_results_path.write_text(json.dumps(gc_results, indent=2))

# Deduplicate by data hash
unique_entities = set()
unique_endorsed = set()
for r in gc_results:
    if r.get('entities_data_hash'):
        unique_entities.add(r['entities_data_hash'])
    if r.get('endorsed_data_hash'):
        unique_endorsed.add(r['endorsed_data_hash'])

print(f'\n{"=" * 70}')
print(f'RESULTS')
print(f'  Pairs converted:    {sum(1 for r in gc_results if r["status"] in ("ok", "cached"))}')
print(f'  Pairs failed:       {sum(1 for r in gc_results if r["status"] == "failed")}')
print(f'  Unique entities:    {len(unique_entities)}')
print(f'  Unique endorsed:    {len(unique_endorsed)}')
print(f'  GC output:          {gc_dir}')
print(f'  Results log:        {gc_results_path}')

## Summary

After running Phase 1, check the output on Google Drive:

```
My Drive/ubl-gc-revisions/drive-revision-pairs/
├── drive-revisions-metadata.json   ← Full revision lists + metadata
├── temporal-pairs.json             ← Just the computed pairs (smaller)
├── ubl25-ods/                      ← (Phase 2) Downloaded ODS files
│   ├── lib-rev-{id}.ods
│   └── doc-rev-{id}.ods
├── ubl25-gc/                       ← (Phase 3) Converted GC files
│   ├── lib{id}-doc{id}.entities.gc.gz
│   └── lib{id}-doc{id}.endorsed.gc.gz
├── ubl25-download-log.json         ← (Phase 2) Download details
└── ubl25-gc-results.json           ← (Phase 3) Conversion results
```

**Key file for Claude:** Upload `temporal-pairs.json` to the Claude Code
session. It contains the timestamp-paired revision IDs that dramatically
reduce the search space.

If you also ran Phase 2+3, upload `ubl25-gc-results.json` for the
content-hash deduplication results.